In [1]:
import torch
import numpy as np
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


In standard AI (Keras/sklearn), you pick from a menu: MSE, CrossEntropy, MAE. In Research, you often invent the menu. You might need a loss that:

Penalizes "false positives" 10x more than "false negatives" (Medical AI).

Ignores errors if the value is below a certain threshold (Robust Regression).

Forces the model to output sparse vectors.

## Method 1: The Functional Approach (The 90% Use Case)
If your loss doesn't have internal parameters (weights) to learn, just write a Python function.

Scenario: We want a "Penalized Overshoot Loss".

If the model predicts too low, it's a normal error.

If the model predicts too high, we penalize it double. (Imagine predicting stock prices where buying too high is fatal).

In [2]:
import torch

def penalized_overshoot_loss(output, target):
    # 1. Calculate raw difference
    error = output - target
    
    # 2. Define penalty
    # If error > 0 (Overshoot), multiply by 2. Else multiply by 1.
    # We use torch.where (like np.where)
    penalty_mask = torch.where(error > 0, 2.0, 1.0)
    
    # 3. Calculate squared error weighted by penalty
    loss = (error ** 2) * penalty_mask
    
    # 4. Return the mean (scalar)
    return loss.mean()

# Usage
pred = torch.tensor([10.0, 5.0], requires_grad=True)
true = torch.tensor([8.0, 6.0]) # True values

loss = penalized_overshoot_loss(pred, true)
loss.backward()

print(loss.item())

4.5


## Method 2: The nn.Module Approach (The Professional Way)
Researchers usually wrap losses in a class inheriting nn.Module.

Why? It integrates with the PyTorch API (saving/loading).

Why? You can store hyperparameters (like margin or threshold) inside the class.

Scenario: Implementing MAE (Mean Absolute Error) manually as a class.

In [3]:
import torch.nn as nn

class CustomMAE(nn.Module):
    def __init__(self):
        super(CustomMAE, self).__init__()
        
    def forward(self, output, target):
        # abs() is differentiable everywhere except 0 (PyTorch handles subgradients)
        return torch.abs(output - target).mean()

# Usage in a training loop
criterion = CustomMAE()
# loss = criterion(outputs, labels)

### Huber loss
##### DON'T DO THIS:
```
if abs_error < delta:
    return quadratic
else:
    return linear

```
This fails because abs_error is a tensor of 32 or 64 numbers (the batch).

In [4]:
import torch
import torch.nn as nn

class CustomHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super(CustomHuberLoss, self).__init__()
        self.delta = delta

    def forward(self, output, target):
        # 1. Calculate absolute error |x - y|
        # We use abs() because the formula depends on magnitude
        abs_error = torch.abs(output - target)
        
        # 2. Calculate the Quadratic Component (MSE-like) for small errors
        # Formula: 0.5 * (x - y)^2
        quadratic = 0.5 * (output - target) ** 2
        
        # 3. Calculate the Linear Component (MAE-like) for large outliers
        # Formula: delta * (|x - y| - 0.5 * delta)
        linear = self.delta * (abs_error - 0.5 * self.delta)
        
        # 4. Selector
        # torch.where(condition, if_true, if_false)
        # If error is small (< delta), pick quadratic. Else pick linear.
        loss = torch.where(abs_error < self.delta, quadratic, linear)
        
        # 5. Reduction (Return mean scalar)
        return loss.mean()

# --- Testing the Implementation ---

# Create dummy data
# pred[0] is close to target (Small error -> Quadratic)
# pred[1] is far from target (Large outlier -> Linear)
predictions = torch.tensor([1.5, 10.0], requires_grad=True)
targets = torch.tensor([1.0, 1.0])

# Instantiate
criterion = CustomHuberLoss(delta=1.0)

# Forward Pass
loss = criterion(predictions, targets)

print(f"Loss Value: {loss.item():.4f}")

# Backward Pass (Proving the graph works)
loss.backward()
print(f"Gradients on input: {predictions.grad}")

Loss Value: 4.3125
Gradients on input: tensor([0.2500, 0.5000])
